# DiT generation HPO: search and seed confirmation

Optimize the standalone conditional **diffusion_transformer** on CIFAR-10 through
the codebase's existing HPO API. The scalar objective is post-training validation
**EMA noise_loss**, minimized. The classifier branch is absent.

Protocol: inspect the first 100 successful trials, reach **200**, optionally
extend in 50-trial stages to **300**, then rerun the top **three** configurations
on **three fresh paired seeds**. The first **40** search observations are random;
TPE chooses subsequent configurations. Every search and confirmation run uses
the same **50-epoch maximum** and **early-stopping patience 5**.

The notebook is delivered unexecuted. Search and confirmation cells launch real
training when you execute them. Run cells individually to review each stage.
No campaign deadline is imposed. Failures count against the separate 400-attempt
ceiling, but only finite COMPLETE trials satisfy the successful-trial target.

## Remote execution

Open this notebook **directly in a fresh Python kernel on a supplied remote
H100/A100 container**. The kernel coordinates child processes and must not import
TensorFlow or Keras. Do not wrap it in the older run_notebooks.py campaign runner:
that wrapper already owns a GPU lease.

The helper verifies the active host/GPU UUID, checkout source hashes and package
versions, queues for the deployed two-slot allocator, reserves 13312 MiB, and
caps each worker's TensorFlow allocation at 12288 MiB. Unknown ownership fails
closed. A worker releases its memory by exiting before the coordinator releases
its lease. Existing campaign jobs are preserved.

Required runtime: TensorFlow **2.20.0**, Keras **3.11.2**, plus the repository's
supporting dependencies. This notebook never installs packages or starts Docker.
All computation, including metadata review, must run on the remote container.

In [ ]:
import os
import platform
import sys

from pathlib import Path


if platform.system() != "Linux":
    raise RuntimeError("Use the supplied remote GPU containers; laptop execution is prohibited.")

CHECKOUT_ROOT = Path("/workspace/Continual-Learning-with-Diffusion-Vision-Transformers-main")
if not (CHECKOUT_ROOT / "common" / "dit_hpo_runner.py").is_file():
    raise RuntimeError("Sync this notebook and its three common/dit_hpo_*.py helpers to the remote checkout.")

os.chdir(CHECKOUT_ROOT)
if str(CHECKOUT_ROOT) not in sys.path:
    sys.path.insert(0, str(CHECKOUT_ROOT))

# The coordinator performs no TensorFlow work; admitted children expose their GPU.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ.setdefault("KERAS_HOME", "/workspace/.keras")

from common.dit_hpo_remote import inspect_remote
from common.dit_hpo_runner import (
    make_plan, search_summary, run_search, freeze_finalists, 
    run_confirmations, confirmation_summary
)


runtime = inspect_remote(CHECKOUT_ROOT)
display({key: value for key, value in runtime.items() if key != "source_sha256"})
print("Fingerprinted source files:", len(runtime["source_sha256"]))

In [ ]:
DATASET = "CIFAR10"
RESULTS_PATH = "files/results/dit_generation_hpo_v1"
EPOCHS = 50
N_STARTUP_TRIALS = 40
SEARCH_SEED = 42

REVIEW_TARGET = 100
SEARCH_TARGET = 200
EXTENSION_TARGET = 200  # Set 250 or 300 after reviewing the search, before freezing finalists.
MAX_ATTEMPTS = 400
BATCH_TRIALS = 10

TOP_K = 3
CONFIRMATION_SEEDS = [101, 202, 303]

plan = make_plan(
    CHECKOUT_ROOT, 
    RESULTS_PATH, 
    dataset_name=DATASET, 
    epochs=EPOCHS, 
    n_startup_trials=N_STARTUP_TRIALS, 
    seed=SEARCH_SEED
)
display(plan["hpo"])
print("Persistent study:", plan["study_root"])

## Search space

The engine uses the maintained common.hpo suggestions; this table documents the
default space rather than reimplementing it.

| Group | Parameters and choices |
| --- | --- |
| Capacity | dim 32/64/96/128, always 4 attention heads |
| Patches | size 2/4 when divisible; CNN patchification off/on |
| Backbone | plain or compact U-DiT when patch grid permits |
| Plain depth | 2/3/4/5/6 |
| U-DiT | fixed nine stages; resampling positions 2d_sincos/new_weight |
| Transformer | MLP ratio 2/4; drop-path 0/0.05/0.1; refiner CNN off/on |
| Optimization | Adam/AdamW; learning rate 3e-4 to 5e-3, log-uniform |
| AdamW only | weight decay 1e-6 to 1e-3, log-uniform |
| Batching | 32/64/128 |
| Clipping | per-variable None/0.5/1/5; global None/0.5/1/5 only if per-variable is None |
| Diffusion | 500/1000 timesteps; clipped_cosine/squaredcos_cap_v2/linear |
| Conditioning | p_uncond 0.05/0.1/0.2 |
| EMA | decay 0.995/0.999 |
| Auxiliary image loss | coefficient 0/0.01/0.05/0.1 |

MSE, ordinary-fit cosine learning-rate decay, EMA evaluation, CFG support and
adaptive normalization are fixed. Wrapper sampling uses 50 steps, CFG 4, eta 0;
final image reports use CFG 3. These sampling settings are not optimized.
This study uses no distillation, reconstruction-mode override, or continual replay.

## HPO engine and recovery

Each stage below calls **run_search**, which launches registered child processes
that invoke **common.hpo.run_hpo** with the settings printed above. The public
API's total-budget mode bounds allocated attempts. The coordinator counts
finite COMPLETE trials separately and supplies additional bounded attempts if
OOM failures or pruned nonfinite objectives prevent reaching the target.

Each child handles at most ten new trial allocations, then exits. The existing
SQLite study, sampler RNG state, study specification, YAML configurations,
training histories and reports persist. Re-executing a reached stage adds no
trials. An interrupted search uses the engine's existing recovery protocol;
no new coordinator bypasses the study lock.

Use a **new RESULTS_PATH** after changing the dataset, training budget, startup
budget, seed, source implementation or validation protocol. Trial targets can
increase without changing the scientific recipe. No official test-set metric
is used for selection.

In [ ]:
display(search_summary(plan))

In [ ]:
first_stage = run_search(
    plan, 
    target_completed=REVIEW_TARGET, 
    max_attempts=MAX_ATTEMPTS, 
    batch_trials=BATCH_TRIALS
)
if not first_stage["target_reached"]:
    raise RuntimeError("Attempt ceiling reached before the review target; inspect failed-trial logs.")

## Review after 100 successful trials

Inspect failures, coverage of plain/U-DiT, the best validation loss, and trial
durations. Do not remove a branch only because a few random trials performed
poorly. The next stage keeps the same data split and training/evaluation budget.

In [ ]:
import optuna


study = optuna.load_study(
    study_name=plan["study_name"], 
    storage="sqlite:///" + (Path(plan["study_root"]) / "study.db").as_posix()
)
trial_table = study.trials_dataframe()
display(trial_table.sort_values("value").head(15))
display(search_summary(plan))

In [ ]:
main_stage = run_search(
    plan, 
    target_completed=SEARCH_TARGET, 
    max_attempts=MAX_ATTEMPTS, 
    batch_trials=BATCH_TRIALS
)
if not main_stage["target_reached"]:
    raise RuntimeError("Attempt ceiling reached before the main target; inspect the search status.")

## Optional extension

Review the best-loss improvement over the last 50 successful trials and branch
coverage. These summaries are descriptive; they do not prove convergence.
EXTENSION_TARGET=200 makes the next cell an idempotent status check. Set it to
250 or 300 only if you choose to continue, **before freezing finalists**.
Fresh-seed confirmation provides a separate check on ranking stability.

In [ ]:
if EXTENSION_TARGET not in (200, 250, 300) or EXTENSION_TARGET < SEARCH_TARGET:
    raise ValueError("The extension must preserve the search budget and be 200, 250 or 300.")
final_stage = run_search(
    plan, 
    target_completed=EXTENSION_TARGET, 
    max_attempts=MAX_ATTEMPTS, 
    batch_trials=BATCH_TRIALS
)
if not final_stage["target_reached"]:
    raise RuntimeError("Resolve the incomplete search target before selecting finalists.")
display(final_stage)

## Freeze and confirm three finalists

The next cell freezes the three best distinct configurations, copies their
**original input configurations**, and records hashes and the paired seed list.
After selection is frozen this runner will not extend the search.

Confirmation uses public load_config/get_datasets/get_model/train_model/report
APIs. It first constructs the original train/validation split and shuffle stream
using SEARCH_SEED, then initializes a new model with each fresh training seed.
It never restores the candidate's trained weights. The sampled architecture,
optimizer, losses, epoch limit and early-stopping rule remain unchanged.

The same three seeds are used for every finalist. Training and final stochastic
evaluation vary by confirmation seed; this is not a decomposition of individual
noise sources. Completed repeats are authenticated and skipped on rerun.
Interrupted attempts are retained and a fresh attempt directory is used.

In [ ]:
finalists = freeze_finalists(plan, CONFIRMATION_SEEDS, top_k=TOP_K)
display(finalists)

In [ ]:
confirmation_records = run_confirmations(plan)
print("Completed confirmation runs:", len(confirmation_records))

In [ ]:
import pandas as pd


summary = pd.DataFrame(confirmation_summary(plan))
display(summary.sort_values("mean_noise_loss"))
if not summary["all_seeds_complete"].all():
    raise RuntimeError("Some confirmation repeats are incomplete; do not select from partial means.")
summary.to_csv(Path(plan["control_root"]) / "confirmation_summary.csv", index=False)

## Interpretation and saved artifacts

Compare mean validation noise_loss and sample standard deviation across the
three paired seeds. Inspect individual receipts and histories when candidate
differences are small. Three repeats are a practical stability check, not proof
that one configuration is universally better.

The study optimizes a denoising proxy. Image quality, diversity and downstream
replay usefulness require separate evaluation after configuration selection.
Keep the official test set untouched until the final evaluation plan is fixed.

The study directory contains study.db, trials.csv, configs and runs.
notebook_runner contains recipe.json, search_status.json, worker requests/logs,
finalists.json, frozen YAML inputs, confirmation receipts and summary tables.
Partial/failed artifacts are retained. All notebook outputs are initially empty.